# GRPO & Verifiable Rewards

Two ideas that arrived together and reshaped LLM post-training:

**RLVR** — *reinforcement learning from verifiable rewards*. Instead of a learned reward
model approximating human preference, use a reward you can **check**: does the code pass
its tests, does the maths answer match, does the output parse. A checkable reward cannot
be gamed in the way a learned one can, because there is no approximation to exploit.

**GRPO** — *Group Relative Policy Optimization*. Sample a group of `G` completions for the
same prompt and use their **relative** rewards as the advantage. The group average *is*
the baseline, so the critic disappears entirely — halving the memory and removing an
entire model that could be wrong.

They fit together naturally: verifiable rewards are cheap to evaluate many times, which is
exactly what sampling a group requires.

Follows [PPO from Scratch](ppo-from-scratch.ipynb); contrast with
[Reward Models & Reward Hacking](reward-models-and-hacking.ipynb).

## 1. What & Why

[PPO](ppo-from-scratch.ipynb) needs a critic to compute advantages, and in LLM RLHF that
critic is a second network the size of the policy. It must be trained alongside, it is
frequently inaccurate early on, and it doubles the memory for optimiser state.

GRPO's observation: for a *single prompt* answered `G` times, you already have everything
you need to know whether a completion was better than average — the other `G−1`
completions.

```
A_i = (r_i − mean(r_1..r_G)) / std(r_1..r_G)
```

That is the whole advantage estimate. No value network, no GAE, no bootstrapping. The
group *is* the baseline, and it is an unbiased one by construction.

**Why verifiable rewards matter here.** A learned reward model gives a dense but
approximate score; optimising hard against it produces
[reward hacking](reward-models-and-hacking.ipynb). A verifiable reward is sparse — often
just 0 or 1 — but it is *correct*. You cannot Goodhart a unit test in the way you can
Goodhart a preference model, because there is no gap between the proxy and the objective:
the test **is** the objective.

**The trade** is sample efficiency. A binary reward carries far less information per
sample than a scalar preference score, and if every completion in a group fails, the
advantage is zero for all of them and you learn nothing from that prompt.

## 2. Mental Model

**Grading on a curve, one question at a time.**

A critic is an examiner who tries to predict, before seeing any answers, what a typical
answer to this question is worth. That is a hard prediction, it needs its own training,
and when it is wrong every advantage computed from it is wrong.

GRPO grades on a curve instead: collect `G` answers to the *same* question, and score each
against the group's own average. No prediction needed — the comparison set is right there.

Three consequences follow directly:

- **The prompt's intrinsic difficulty cancels out.** A hard prompt where everyone scores
  low and an easy one where everyone scores high both produce advantages centred on zero.
  This is the same variance reduction as a perfect state-dependent baseline, obtained for
  free.
- **A unanimous group teaches nothing.** If all `G` completions get the same reward, every
  advantage is zero. All-correct and all-wrong prompts contribute no gradient — which
  makes *prompt difficulty selection* a first-class concern rather than an afterthought.
- **`G` controls estimate quality.** Small groups give noisy baselines; large groups cost
  linearly more sampling. This is the algorithm's central tuning knob.

## 3. Key Concepts

| Term | What it means |
|---|---|
| **RLVR** | RL from rewards you can programmatically verify — unit tests, exact-match answers, parsers, proofs. |
| **Group** | The `G` completions sampled for one prompt. Typically 4–64. |
| **Group-relative advantage** | `(r_i − mean) / std` within the group. Replaces the critic entirely. |
| **Critic-free** | No value network. Roughly halves memory and removes a whole training problem. |
| **Sparse reward** | Usually binary. Correct and cheap, but low information per sample. |
| **Degenerate group** | All completions get the same reward → zero advantage → no gradient from that prompt. |
| **Curriculum / difficulty filtering** | Selecting prompts where the model succeeds *sometimes*, since those are the only ones producing signal. |
| **KL to reference** | Still used, to stop the policy drifting into degenerate text that happens to pass the checker. |
| **Format reward** | A small shaped reward for well-formed output, so early training has any signal at all. |
| **Reward hacking under RLVR** | Not gaming the reward *model*, but gaming the *checker*: hard-coding test cases, exploiting a weak parser. |

## 4. Setup

NumPy. The point of the examples is to compare GRPO's advantage estimator against a
critic-based one under conditions we control — particularly when the critic is wrong,
which is the case that matters.

In [1]:
# %pip install numpy

import numpy as np

rng = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.1


## 5. Worked Examples

### Example 1 — group-relative advantage, in six lines

The entire estimator. Note what it does to prompts of different difficulty.

In [2]:
def group_advantage(rewards, eps=1e-8):
    '''GRPO's advantage: standardise the rewards WITHIN the group.'''
    r = np.asarray(rewards, dtype=float)
    return (r - r.mean()) / (r.std() + eps)

groups = {
    "mixed (ideal)":        [1, 0, 1, 0, 0, 1, 0, 0],
    "mostly correct":       [1, 1, 1, 1, 1, 1, 0, 1],
    "all correct":          [1, 1, 1, 1, 1, 1, 1, 1],
    "all wrong":            [0, 0, 0, 0, 0, 0, 0, 0],
    "graded (partial credit)": [0.9, 0.4, 1.0, 0.2, 0.6, 0.5, 0.0, 0.8],
}
print(f"{'group':26} {'mean r':>7} {'advantages':>46} {'|grad|':>8}")
for name, rew in groups.items():
    adv = group_advantage(rew)
    print(f"{name:26} {np.mean(rew):7.2f} {str(np.round(adv, 2)):>46} "
          f"{np.abs(adv).sum():8.2f}")

print("\nThe first group is the useful one: some completions beat the group average and")
print("are reinforced, some fall below it and are suppressed.")
print("\n'all correct' and 'all wrong' produce ZERO advantage everywhere -- the sampling")
print("cost was paid and no gradient results. That is GRPO's defining practical")
print("constraint: you learn only from prompts the model gets right SOMETIMES.")
print("\nNote also that a graded reward spreads the advantages more evenly than a binary")
print("one, which is why partial credit is worth engineering when you can verify it.")

group                       mean r                                     advantages   |grad|
mixed (ideal)                 0.38 [ 1.29 -0.77  1.29 -0.77 -0.77  1.29 -0.77 -0.77]     7.75
mostly correct                0.88 [ 0.38  0.38  0.38  0.38  0.38  0.38 -2.65  0.38]     5.29
all correct                   1.00                      [0. 0. 0. 0. 0. 0. 0. 0.]     0.00
all wrong                     0.00                      [0. 0. 0. 0. 0. 0. 0. 0.]     0.00
graded (partial credit)       0.55 [ 1.08 -0.46  1.39 -1.08  0.15 -0.15 -1.7   0.77]     6.79

The first group is the useful one: some completions beat the group average and
are reinforced, some fall below it and are suppressed.

'all correct' and 'all wrong' produce ZERO advantage everywhere -- the sampling
cost was paid and no gradient results. That is GRPO's defining practical
constraint: you learn only from prompts the model gets right SOMETIMES.

Note also that a graded reward spreads the advantages more evenly than a binary
one

### Example 2 — GRPO versus a critic, when the critic is wrong

The comparison that justifies dropping the value network. A critic must *predict* the
expected reward; early in training that prediction is poor, and every advantage inherits
its error.

In [3]:
def simulate(n_prompts=4000, G=8, critic_bias=0.0, critic_noise=0.0, seed=0):
    r = np.random.default_rng(seed)
    # Prompts vary a lot in difficulty -- this is what a critic has to model.
    difficulty = r.beta(2, 2, n_prompts)
    grpo_adv, critic_adv, true_adv = [], [], []
    for p in range(n_prompts):
        p_success = difficulty[p]
        rewards = (r.random(G) < p_success).astype(float)
        # ground truth: how much better than this prompt's TRUE expectation
        truth = rewards - p_success
        grpo_adv.append(group_advantage(rewards) * rewards.std())   # undo standardisation
        critic_pred = p_success + critic_bias + r.normal(0, critic_noise)
        critic_adv.append(rewards - critic_pred)
        true_adv.append(truth)
    return (np.concatenate(true_adv), np.concatenate(grpo_adv),
            np.concatenate(critic_adv))

print(f"{'critic quality':28} {'GRPO err':>10} {'critic err':>12} {'winner':>9}")
for label, bias, noise in [("perfect critic", 0.0, 0.0),
                           ("slightly noisy", 0.0, 0.10),
                           ("noisy", 0.0, 0.25),
                           ("biased +0.2", 0.2, 0.10),
                           ("bad (early training)", 0.3, 0.35)]:
    truth, g, c = simulate(critic_bias=bias, critic_noise=noise, seed=7)
    ge = float(np.sqrt(np.mean((g - truth) ** 2)))
    ce = float(np.sqrt(np.mean((c - truth) ** 2)))
    print(f"{label:28} {ge:10.4f} {ce:12.4f} {'critic' if ce < ge else 'GRPO':>9}")

print("\nA PERFECT critic beats GRPO -- it knows the prompt's true expectation, while")
print("GRPO must estimate it from G samples. That is the honest baseline.")
print("\nBut a perfect critic is exactly what you do not have. As soon as the critic is")
print("biased or noisy -- which describes every critic early in training -- the group")
print("estimate wins, and it needs no second network, no extra optimiser state, and no")
print("separate training loop that can itself diverge.")

critic quality                 GRPO err   critic err    winner
perfect critic                   0.1589       0.0000    critic
slightly noisy                   0.1589       0.1001    critic
noisy                            0.1589       0.2503      GRPO


biased +0.2                      0.1589       0.2240      GRPO
bad (early training)             0.1589       0.4620      GRPO

A PERFECT critic beats GRPO -- it knows the prompt's true expectation, while
GRPO must estimate it from G samples. That is the honest baseline.

But a perfect critic is exactly what you do not have. As soon as the critic is
biased or noisy -- which describes every critic early in training -- the group
estimate wins, and it needs no second network, no extra optimiser state, and no
separate training loop that can itself diverge.


### Example 3 — group size is the knob

`G` trades sampling cost against advantage quality. The relationship is the familiar
`1/√G`, with a hard floor at `G = 1`.

In [4]:
print(f"{'G':>4} {'adv RMSE vs truth':>19} {'degenerate groups':>19} {'samples/prompt':>15}")
for G in (1, 2, 4, 8, 16, 32, 64):
    truth, g, _ = simulate(n_prompts=3000, G=G, seed=11)
    err = float(np.sqrt(np.mean((g - truth) ** 2)))
    # how often does the whole group agree, yielding no gradient?
    r = np.random.default_rng(11)
    diff = r.beta(2, 2, 3000)
    degen = float(np.mean([len(set((r.random(G) < d).astype(float))) == 1 for d in diff]))
    print(f"{G:>4} {err:19.4f} {degen:19.1%} {G:15d}")

print("\nG=1 is useless: a group of one has zero variance, so the advantage is undefined")
print("(and every group is degenerate). GRPO fundamentally requires G >= 2.")
print("\nThe error falls with G, but so does the fraction of wasted prompts -- with G=4,")
print("a large share of groups are unanimous and contribute nothing. That second column")
print("is the one people forget when picking G: small groups do not just give noisy")
print("advantages, they throw away whole prompts.")
print("\nTypical production values are 8-64, and the choice is usually bounded by")
print("sampling throughput rather than by the statistics.")

   G   adv RMSE vs truth   degenerate groups  samples/prompt
   1              0.4515              100.0%               1
   2              0.3136               58.5%               2
   4              0.2246               29.5%               4


   8              0.1563               10.0%               8


  16              0.1128                3.1%              16
  32              0.0774                0.8%              32
  64              0.0570                0.3%              64

G=1 is useless: a group of one has zero variance, so the advantage is undefined
(and every group is degenerate). GRPO fundamentally requires G >= 2.

The error falls with G, but so does the fraction of wasted prompts -- with G=4,
a large share of groups are unanimous and contribute nothing. That second column
is the one people forget when picking G: small groups do not just give noisy
advantages, they throw away whole prompts.

Typical production values are 8-64, and the choice is usually bounded by
sampling throughput rather than by the statistics.


### Example 4 — verifiable rewards do not eliminate reward hacking, they relocate it

The claim "you cannot game a unit test" is too strong. You cannot game the *reward model*
because there isn't one — but you can absolutely game a **weak checker**.

In [5]:
# A verifier that checks the final answer appears in the output. Cheap, and exploitable.
def weak_checker(output, answer):
    return float(str(answer) in output)

def strict_checker(output, answer):
    '''Parse a single final answer from a required format, and compare exactly.'''
    import re
    m = re.search(r"<answer>\s*(-?\d+)\s*</answer>", output)
    return float(m is not None and m.group(1) == str(answer))

ANSWER = 42
candidates = {
    "correct reasoning":      "8 * 5 = 40, plus 2 gives <answer>42</answer>",
    "correct, wrong format":  "The answer is 42",
    "shotgun every number":   " ".join(str(i) for i in range(100)),
    "restate the question":   "Is the answer 42? Let me think... <answer>42</answer>",
    "wrong answer":           "<answer>37</answer>",
}
print(f"{'completion':26} {'weak checker':>14} {'strict checker':>16}")
for name, out in candidates.items():
    print(f"{name:26} {weak_checker(out, ANSWER):14.0f} "
          f"{strict_checker(out, ANSWER):16.0f}")

print("\nThe 'shotgun' completion contains no reasoning and passes the weak checker")
print("perfectly. A policy optimised against it will learn to emit number soup -- and")
print("the reward curve will look excellent throughout.")
print("\nThe strict checker resists that particular exploit, and rejects the correctly-")
print("reasoned but badly-formatted answer, which is the cost of strictness.")

# What the policy learns is entirely determined by which checker you used.
print("\nreward a policy would collect under each checker, if it learned to shotgun:")
shotgun = " ".join(str(i) for i in range(100))
for name, checker in [("weak", weak_checker), ("strict", strict_checker)]:
    scores = [checker(shotgun, a) for a in rng.integers(0, 100, 500)]
    print(f"  {name:8}: {np.mean(scores):.1%} of problems 'solved' by emitting 0-99")

print("\nSo RLVR's guarantee is narrower than it sounds: the reward is exactly what you")
print("specified, and any gap between your CHECKER and the capability you meant will be")
print("found and exploited. The engineering effort moves from reward-model training to")
print("checker design -- which is more tractable, and not free.")

completion                   weak checker   strict checker
correct reasoning                       1                1
correct, wrong format                   1                0
shotgun every number                    1                0
restate the question                    1                1
wrong answer                            0                0

The 'shotgun' completion contains no reasoning and passes the weak checker
perfectly. A policy optimised against it will learn to emit number soup -- and
the reward curve will look excellent throughout.

The strict checker resists that particular exploit, and rejects the correctly-
reasoned but badly-formatted answer, which is the cost of strictness.

reward a policy would collect under each checker, if it learned to shotgun:
  weak    : 100.0% of problems 'solved' by emitting 0-99
  strict  : 0.0% of problems 'solved' by emitting 0-99

So RLVR's guarantee is narrower than it sounds: the reward is exactly what you
specified, and any gap 

## 6. Gotchas & Pitfalls

- **Ignoring degenerate groups.** Examples 1 and 3. Prompts the model always or never
  solves cost full sampling budget and produce no gradient. Filter or curriculum them, and
  monitor the fraction — it is a leading indicator of a stalling run.
- **Setting `G` too small.** Below ~4 the baseline is noisy and most groups are unanimous.
- **A weak checker.** Example 4. Under RLVR, the checker *is* the objective, and any
  loophole becomes the policy's optimum.
- **Dropping the KL penalty because "verifiable rewards can't be hacked".** The policy can
  still drift into degenerate text that satisfies the checker. Keep a reference-KL term.
- **Standardising by `std` when it is near zero.** A group with almost-identical rewards
  produces enormous advantages after division. Use the epsilon, and consider skipping such
  groups entirely.
- **Binary-only rewards with no shaping.** If the model starts at ~0% success, every group
  is unanimous and nothing happens. A small format reward gives early training something
  to climb.
- **Comparing GRPO to PPO with a well-tuned critic and concluding GRPO is worse.** The
  comparison that matters includes the cost and risk of *training* that critic (Example 2).
- **Assuming verifiable means objective.** A unit test encodes someone's belief about what
  the code should do. RLVR moves the subjectivity, it does not remove it.

## 7. When to Use vs Alternatives

| Situation | Reach for |
|---|---|
| Maths, code, anything checkable | **GRPO + RLVR** — the current default for reasoning training |
| Open-ended quality, no checker possible | [**PPO**](ppo-from-scratch.ipynb) with a [reward model](reward-models-and-hacking.ipynb) |
| Preference pairs, no RL infrastructure | **DPO** — closed-form, no sampling loop |
| Sampling is very expensive | **PPO** — reuses each rollout for several epochs; GRPO needs `G` samples per prompt |
| A trustworthy critic already exists | **PPO** — Example 2's first row is real |
| Mixed verifiable and subjective objectives | Both, combined: verifiable reward plus a preference model, with separate weights |

**The honest position.** GRPO is not a better optimiser than PPO — it is PPO with the
critic replaced by a group baseline. That trade wins when the critic is hard to train and
sampling is cheap, which is exactly the regime of verifiable-reward reasoning tasks, and
it is why the combination took over that niche so quickly.

The deeper shift is RLVR rather than GRPO. Moving from *learned* to *checkable* rewards
removes the reward-hacking failure mode that dominates RLHF — and replaces it with
checker design, which is a better problem to have because a checker's failures are
inspectable in a way a reward model's are not.

## 8. Resources

- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning](https://arxiv.org/abs/2402.03300) — introduces GRPO; Section 4 is the algorithm and its motivation.
- [DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning](https://arxiv.org/abs/2501.12948) — GRPO + verifiable rewards at scale, including the format-reward trick.
- [Let's Verify Step by Step](https://arxiv.org/abs/2305.20050) — process versus outcome supervision; the case for richer verification signals.
- [The Alignment Handbook](https://github.com/huggingface/alignment-handbook) and [TRL's GRPOTrainer](https://huggingface.co/docs/trl/grpo_trainer) — working implementations.
- [Open-R1](https://github.com/huggingface/open-r1) — an open reproduction; see also [the Open-R1 notebook](../03-llm-inference-training-optimization/open-r1.ipynb).
- [Tulu 3: Pushing Frontiers in Open Language Model Post-Training](https://arxiv.org/abs/2411.15124) — RLVR applied across a broad task mixture, with ablations on what verifies well.
- [Reward Hacking in Reinforcement Learning](https://lilianweng.github.io/posts/2024-11-28-reward-hacking/) — a survey that covers the checker-gaming failure of Example 4.